# GENERATE BURDEN SCORE

In [ ]:
# Load packages 
import os

# Set bucket environment
my_bucket = os.getenv('WORKSPACE_BUCKET')

# COMMAND 

In [ ]:
%%writefile gene_burden_analysis.sh
#!/bin/bash

# Exit on any error or unset variable
set -o errexit
set -o nounset

# Make unmatched globs expand to nothing 
shopt -s nullglob

# Loop through populations
for pop in EUR; do
    echo "Processing population: ${pop}"
    
    # Check that cohort file exists
    if [ ! -f "${cohort}/${pop}-like_exome_qc.txt" ]; then
        echo "Cohort file for ${pop} not found!"
        exit 1
    fi
    
    # Prepare PLINK2 binary
    cp "${plink2}" plink2
    chmod +x plink2

    echo "Number of samples in ${pop}:"
    wc -l "${cohort}/${pop}-like_exome_qc.txt"
    
    # === GENE BURDEN ANALYSIS ===
    echo "Running gene-level burden PRS for chromosome ${chrom}..."

    for gene_burden in ${gene_weights}/HC-burden-prs-*-chr${chrom}.txt; do
        echo "Processing ${gene_burden} ...."

        awk -F '\t' '{print $3}' "${gene_burden}" > "gene_${chrom}.snplist.txt"

        ./plink2 \
            --bfile "${genos}/exome.chr${chrom}" \
            --keep "${cohort}/${pop}-like_exome_qc.txt" \
            --extract "gene_${chrom}.snplist.txt" \
            --memory 3100 \
            --max-maf 0.01 \
            --mac 1 \
            --score "${gene_burden}" 3 5 6 no-mean-imputation cols=+nallele,+scoresums \
            --out "${gene_output}/${pop}_$(basename "${gene_burden}" .txt)" \
            || { 
            echo "$(basename "${gene_burden}")" >> "${gene_output}/failed_genes.txt";
            echo "PLINK failed for $(basename "${gene_burden}") — skipping.";
        }
    done

    # === EXOME BURDEN ANALYSIS ===
    echo "Running exome-wide burden PRS for chromosome ${chrom}..."
    awk -F '\t' '{print $3}' "${exome_weights}/Exome-burden-prs-chr${chrom}.txt" > exome.snplist.txt

    ./plink2 \
        --bfile "${genos}/exome.chr${chrom}" \
        --keep "${cohort}/${pop}-like_exome_qc.txt" \
        --extract exome.snplist.txt \
        --memory 3100 \
        --max-maf 0.01 \
        --mac 1 \
        --score "${exome_weights}/Exome-burden-prs-chr${chrom}.txt" 3 5 6 no-mean-imputation cols=+nallele,+scoresums \
        --out "${exome_output}/${pop}-Exome-burden-prs-chr${chrom}"

done


In [ ]:
# Copy gene_burden_analysis.sh script to my_bucket 
!gsutil cp gene_burden_analysis.sh {my_bucket}/data/scripts/

In [ ]:
# Check gene_burden_analysis.sh script in my_bucket 
!gsutil cat {my_bucket}/data/scripts/gene_burden_analysis.sh

# SUBMISSION

In [ ]:
%%bash --out burden_analysis

#############################################################################
# DSUB SUBMISSION
#############################################################################

# Source the dsub setup script
source ~/aou_dsub.bash 

# Define chromosome range for processing
LOWER=1
UPPER=23

# Loop through the specified chromosome range
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

# AoU researcher workbench: dsub command specifications
aou_dsub \
  --disk-size 420 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome/" \
  --input-recursive exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores/" \
  --input-recursive gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}/" \
   --output-recursive exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores/" \
  --output-recursive gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}/" \
  --env chrom=${chromo} \
  --env genos="${mnt_path}" \
  --env cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome" \
  --env exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores" \
  --env gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}" \
  --env exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores" \
  --env gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}" \
  --script "${WORKSPACE_BUCKET}/data/scripts/gene_burden_analysis.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################

done